# 07 · 모멘트-곡률 해석

원 문서의 `moment_curvature.ipynb` 에 대응한다.

모멘트-곡률 해석은 **사용** 응력-변형률 관계를 쓰고, 실제 거동을 보는
해석이므로 **강도감소계수를 적용하지 않는다**.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

# 한글 글꼴이 없는 환경에서도 그림이 깨지지 않도록 축 라벨은 ASCII 로 둔다
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96

In [ ]:
from concreteproperties import ConcreteSection
from sectionproperties.pre.library import concrete_rectangular_section

from concreteproperties_kds import KDS


def beam_section(fck=27, fy=400):
    """400 x 600 보 단면 (상부 2-D16, 하부 4-D22, 피복 50 mm)."""
    kds = KDS(column_type="tie")
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=600, b=400,
        dia_top=16, area_top=198.6, n_top=2, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=4, c_bot=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec


def column_section(fck=27, fy=400, column_type="tie"):
    """500 x 500 기둥 단면 (8-D22, 피복 50 mm)."""
    kds = KDS(column_type=column_type)
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=500, b=500,
        dia_top=22, area_top=387.1, n_top=3, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=3, c_bot=50,
        dia_side=22, area_side=387.1, n_side=1, c_side=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec

In [ ]:
kds, conc_sec = beam_section()

mk_res = kds.moment_curvature_analysis(
    theta=0, kappa_inc=1e-7, progress_bar=False
)

kappa = np.array(mk_res.kappa)
moment = np.array(mk_res.m_xy) / 1e6

cracked = kds.calculate_cracked_properties(theta=0)
_, u_res, _ = kds.ultimate_bending_capacity()

print(f"해석 점의 수                 = {len(kappa)}")
print(f"균열모멘트         Mcr       = {cracked.m_cr / 1e6:.2f} kN.m")
print(f"최대 모멘트 (해석) Mmax      = {moment.max():.2f} kN.m")
print(f"극한 휨강도        Mn        = {u_res.m_x / 1e6:.2f} kN.m")
print(f"최대 곡률          kappa_max = {kappa.max():.3e} 1/mm")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.4))
ax.plot(kappa * 1e6, moment, "-")
ax.axhline(
    cracked.m_cr / 1e6, ls="--", color="tab:orange", lw=1,
    label=f"Mcr = {cracked.m_cr / 1e6:.1f}",
)
ax.axhline(
    u_res.m_x / 1e6, ls=":", color="tab:green", lw=1,
    label=f"Mn (ultimate) = {u_res.m_x / 1e6:.1f}",
)
ax.set_xlabel("curvature, kappa (1e-6 /mm)")
ax.set_ylabel("moment (kN.m)")
ax.set_title("Moment-curvature")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

사용 관계로 계산한 최대 모멘트가 등가직사각형 응력블록으로 계산한 극한
휨강도와 0.1 % 이내로 만난다. 서로 다른 두 응력-변형률 관계가 같은 답에
수렴하는 것이다.

## 철근량에 따른 연성

In [ ]:
from concreteproperties import ConcreteSection, add_bar_rectangular_array
from concreteproperties.results import MomentCurvatureResults
from sectionproperties.pre.library import rectangular_section

results = []
labels = []
for n_bar in [3, 5, 8]:
    k = KDS()
    c = k.create_concrete_material(compressive_strength=27)
    s = k.create_steel_material(yield_strength=400)
    g = rectangular_section(d=600, b=400, material=c)
    g = add_bar_rectangular_array(
        geometry=g, area=387.1, material=s,
        n_x=n_bar, x_s=(400 - 2 * 60) / (n_bar - 1),
        anchor=(60, 60), n=16,
    )
    k.assign_concrete_section(ConcreteSection(g))
    results.append(
        k.moment_curvature_analysis(kappa_inc=1e-7, progress_bar=False)
    )
    labels.append(f"{n_bar}-D22")

fig, ax = plt.subplots(figsize=(6.5, 4.4))
for res, lab in zip(results, labels, strict=True):
    ax.plot(np.array(res.kappa) * 1e6, np.array(res.m_xy) / 1e6, label=lab)
ax.set_xlabel("curvature, kappa (1e-6 /mm)")
ax.set_ylabel("moment (kN.m)")
ax.set_title("Effect of reinforcement ratio")
ax.legend()
ax.grid(alpha=0.3)

del MomentCurvatureResults
plt.show()

철근이 많을수록 강도는 커지지만 곡률 연성은 줄어든다.